[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/visualizacion_datos/02_mensaje/comparativa_visual.ipynb)

# Fase 2: Análisis Aclaratorio — Del Hallazgo al Argumento Visual
## Exploratorio vs. Aclaratorio: Ingeniería de la Atención

**Asignatura:** Visualización de Datos · 2026  
**Proyecto:** Landslide ML — Detección de Deslizamientos de Tierra  

---

## Marco conceptual

| | Análisis Exploratorio | Análisis Aclaratorio |
|---|---|---|
| **Audiencia** | Analista de datos | Tomador de decisiones |
| **Propósito** | Descubrir / descartar hipótesis | Comunicar un hallazgo específico |
| **Gráficas** | Rápidas, iterativas | Una gráfica, un mensaje |
| **Herramienta** | Seaborn / Matplotlib | **Plotly `go.Figure()` + `add_annotation()` + `update_layout()`** |

---

### Las 5 preguntas que deben responderse ANTES de diseñar cada gráfica aclaratoria

1. ¿Quién es mi audiencia y cuánto sabe de datos?
2. ¿Cuál es el **único** mensaje que quiero que recuerden?
3. ¿Qué acción quiero que tomen después de ver esto?
4. ¿Cuánto tiempo tienen para procesar la información?
5. ¿Qué objeción anticipada debo refutar con los datos?


In [ ]:
# ── Setup: detectar entorno (Colab vs. local) y configurar rutas ──────────────
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('Landslides_-Applied-ML-Course'):
        os.system('git clone https://github.com/apmontesp/Landslides_-Applied-ML-Course.git')
    DATA_DIR = 'Landslides_-Applied-ML-Course/visualizacion_datos/data'
    FIG_DIR  = 'Landslides_-Applied-ML-Course/visualizacion_datos/data/figures'
else:
    DATA_DIR = '../data'
    FIG_DIR  = '../data/figures'

os.makedirs(FIG_DIR, exist_ok=True)
print(f'Entorno: {"Colab" if IN_COLAB else "Local"}')
print(f'DATA_DIR: {DATA_DIR}')

# ── Importaciones ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio
import json, warnings
warnings.filterwarnings('ignore')

pio.renderers.default = 'notebook'

# Estilo global Matplotlib: fondo blanco, cuadrícula gris tenue
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': '#D1D5DB', 'grid.color': '#E5E7EB',
    'grid.linewidth': 0.8, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
})

# ── Datos ──────────────────────────────────────────────────────────────────────
df_models   = pd.read_csv(f'{DATA_DIR}/comparison_table.csv')
df_channels = pd.read_csv(f'{DATA_DIR}/channel_stats_by_class.csv')

# Convertir columnas con '—' a numérico
for col in ['F1 medio', 'Std', 'AUC-ROC', 'Precisión', 'Recall', 'IoU']:
    df_models[col] = pd.to_numeric(df_models[col], errors='coerce')

df_channels['|Delta|'] = df_channels['Delta'].abs()

print('Librerías y datos cargados OK')
print(f'Modelos: {list(df_models["Modelo"])}')


---
#  Comparativa 1: F1 Score por Modelo

## Paso 0: Las 5 preguntas (ANTES de tocar el código)

| Pregunta | Respuesta |
|---|---|
| **1. Audiencia** | Directivos del proyecto / comité de evaluación técnica. Conocen el concepto de "precisión" pero no los detalles de Random Forest vs. U-Net. |
| **2. Mensaje único** | *"El modelo más simple supera a las redes neuronales profundas en detección de deslizamientos."* |
| **3. Acción deseada** | Adoptar Random Forest como modelo de producción; no invertir en infraestructura GPU adicional. |
| **4. Tiempo disponible** | < 30 segundos. La gráfica debe funcionar como slide en una presentación. |
| **5. Objeción anticipada** | *"¿Cómo puede ser que Deep Learning pierda? No es lo que esperábamos."*  Refutar con los números y el argumento de tamaño del dataset. |

## 1A — Versión Exploratoria (Seaborn · audiencia: analista)

Gráfica rápida para **descubrir** el ranking. Sin pulir, sin mensaje direccionado.

In [ ]:
# ── EXPLORATORIA 1A: Seaborn — rápida, para el analista ─────────────────────
sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(9, 5))
ax.set_facecolor('white')

df_sorted = df_models.sort_values('F1 medio', ascending=False)
colors = ['#2ca02c' if t == 'Clásico' else '#9467bd' for t in df_sorted['Tipo']]

bars = ax.bar(df_sorted['Modelo'], df_sorted['F1 medio'],
              color=colors, alpha=0.8, edgecolor='white')

# Valores encima de cada barra (sin barras de error)
for bar, v in zip(bars, df_sorted['F1 medio']):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.012,
            f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

ax.set_ylim(0, 1.05)
ax.set_ylabel('F1 Score Medio (5-fold CV)')
ax.set_xlabel('Modelo de Machine Learning')
ax.set_title('[EXPLORATORIA] F1 Score por modelo — ¿cuál gana?')
ax.tick_params(axis='x', rotation=20)
ax.yaxis.grid(True, color='#E5E7EB', linewidth=0.8)
ax.xaxis.grid(False)

handles = [mpatches.Patch(facecolor='#2ca02c', label='Clásico'),
           mpatches.Patch(facecolor='#9467bd', label='Deep Learning')]
ax.legend(handles=handles)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exp_1a_f1_exploratorio.png', dpi=120, bbox_inches='tight', facecolor='white')
plt.show()

print('-> Hallazgo: Random Forest (0.837) supera a todos los modelos DL.')
print('-> Señal inesperada: U-Net (0.444), diseñada para segmentación, es la peor.')


## 1B — Versión Aclaratoria (Plotly `go.Figure` · audiencia: directivos)

**Una gráfica, un mensaje:** *"El modelo más simple gana."*

Herramientas Plotly usadas:
- `go.Figure()` + `go.Bar()` — control milimétrico de cada barra
- `fig.add_annotation()` — storytelling integrado en el gráfico
- `fig.update_layout()` — eliminar ruido visual, maximizar data-to-ink ratio

In [ ]:
# ── ACLARATORIA: Plotly go.Figure — control milimétrico ───────────────────────

df_plot = df_models.sort_values('F1 medio', ascending=True)  # ascendente  mejor arriba

# Paleta retórica: UN color de alarma para el hallazgo, gris para el contexto
ROJO   = '#D62728'
GRIS   = '#9CA3AF'
FONDO  = '#FAFAFA'
TEXTO  = '#1F2937'

colores = [ROJO if m == 'Random Forest' else GRIS for m in df_plot['Modelo']]
opacidades = [1.0 if m == 'Random Forest' else 0.45 for m in df_plot['Modelo']]

# ── 1. go.Figure() — la figura base ──────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Bar(
    y=df_plot['Modelo'],
    x=df_plot['F1 medio'],
    orientation='h',
    marker=dict(
        color=colores,
        opacity=opacidades,
        line=dict(width=0),          # sin bordes — data-to-ink
    ),
    error_x=dict(
        type='data',
        array=df_plot['Std'].fillna(0),
        color='#6B7280',
        thickness=2,
        width=6,
    ),
    # Etiqueta solo en Random Forest — el resto no compite
    text=[f'<b>{v:.3f}</b>' if m == 'Random Forest' else ''
          for m, v in zip(df_plot['Modelo'], df_plot['F1 medio'])],
    textposition='outside',
    textfont=dict(color=ROJO, size=14),
    hovertemplate='<b>%{y}</b><br>F1 = %{x:.3f}<extra></extra>',
))

# ── 2. add_annotation() — storytelling directo en el gráfico ─────────────────
# Anotación principal: el mensaje que el directivo debe recordar
fig.add_annotation(
    x=0.65, y='Random Forest',
    text=(
        '<b>⚠️ Hallazgo contraintuitivo</b><br>'
        'Random Forest (F1=0.837) supera<br>'
        'a ResNet-50 en +6.7 puntos.<br>'
        '<i>Con ~3800 muestras, la capacidad<br>'
        'paramétrica de DL es una desventaja.</i>'
    ),
    showarrow=True,
    arrowhead=2,
    arrowcolor=ROJO,
    arrowwidth=2,
    ax=80, ay=0,
    font=dict(size=11, color=ROJO),
    bgcolor='#FEF2F2',
    bordercolor=ROJO,
    borderwidth=1,
    borderpad=6,
    align='left',
)

# Anotación secundaria: refuta la objeción anticipada
fig.add_annotation(
    x=0.25, y='U-Net ResNet-34',
    text='<i>U-Net: arquitectura de segmentación<br>top, pero sin datos suficientes = 0.444</i>',
    showarrow=True,
    arrowhead=1,
    arrowcolor='#6B7280',
    ax=60, ay=30,
    font=dict(size=10, color='#6B7280'),
    bgcolor='white',
    bordercolor='#D1D5DB',
    borderwidth=1,
    borderpad=4,
)

# Línea de referencia con etiqueta — contexto sin distraer
fig.add_vline(
    x=0.80,
    line_dash='dot',
    line_color='#D1D5DB',
    line_width=1.5,
    annotation_text='umbral aceptable (F1=0.80)',
    annotation_font_color='#9CA3AF',
    annotation_font_size=10,
    annotation_position='top right',
)

# ── 3. update_layout() — eliminar ruido, maximizar data-to-ink ───────────────
fig.update_layout(
    # Título como argumento (no descripción)
    title=dict(
        text=(
            '<b>Random Forest supera a las redes neuronales profundas</b><br>'
            '<span style="color:#6B7280;font-size:13px">'
            'Dataset: Landslide4Sense · 3799 muestras · 5-fold CV · F1 Score</span>'
        ),
        font=dict(size=16, color=TEXTO),
        x=0,
        xanchor='left',
    ),
    # Fondo limpio
    paper_bgcolor=FONDO,
    plot_bgcolor=FONDO,
    # Ejes: solo la información necesaria
    xaxis=dict(
        range=[0, 1.05],
        showgrid=False,          # sin grid — data-to-ink
        zeroline=False,
        showline=True,
        linecolor='#E5E7EB',
        tickfont=dict(color='#6B7280', size=11),
        title=dict(text='F1 Score (media · 5-fold CV)', font=dict(size=11, color='#6B7280')),
    ),
    yaxis=dict(
        showgrid=False,
        zeroline=False,
        showline=False,
        tickfont=dict(color=TEXTO, size=12),
    ),
    showlegend=False,            # sin leyenda: el color ya es el mensaje
    height=420,
    margin=dict(l=20, r=20, t=90, b=40),
)

fig.show()
fig.write_html(f'{FIG_DIR}/acl_1b_f1_aclaratorio.html')
print('Gráfica aclaratoria guardada: acl_1b_f1_aclaratorio.html')

###  Justificación de decisiones visuales — Comparativa 1

| Elemento | Exploratoria (Seaborn) | Aclaratoria (Plotly `go`) | Principio |
|---|---|---|---|
| **Color** | Verde=Clásico, Morado=DL (2 categorías) | Rojo=RF, Gris=resto (énfasis único) | Pre-atención: el ojo va al rojo sin procesar el contexto |
| **Grid** | Visible (darkgrid Seaborn) | `showgrid=False` — eliminado | Data-to-ink: el grid no aporta al mensaje |
| **Leyenda** | Presente (necesaria para descubrir) | `showlegend=False` | Una sola categoría importa; leyenda = distracción |
| **Título** | Descriptivo: `"F1 Score por modelo"` | Argumento: `"Random Forest supera a las redes"` | El título ES el mensaje, no el tema |
| **Texto** | Todos los valores etiquetados | Solo RF etiquetado en negrita | Jerarquía: solo lo relevante pide atención |
| **Anotaciones** | Ninguna | 2 `add_annotation()`: hallazgo + refutación | Storytelling integrado en el gráfico |
| **Fondo** | `darkgrid` oscuro | `#FAFAFA` — neutro y limpio | Máximo contraste datos vs. fondo |

---
#  Comparativa 2: Canales Espectrales Discriminativos

## Paso 0: Las 5 preguntas

| Pregunta | Respuesta |
|---|---|
| **1. Audiencia** | Equipo técnico de teledetección. Conocen Sentinel-2 pero no están familiarizados con el análisis de importancia de features en ML. |
| **2. Mensaje único** | *"Solo 2 de 14 bandas satelitales explican casi todo el poder predictivo: RedEdge3 y RedEdge2."* |
| **3. Acción deseada** | Priorizar la adquisición de imágenes Sentinel-2 con bandas RedEdge disponibles (no solo RGB). |
| **4. Tiempo disponible** | ~45 segundos. Slide de soporte en presentación técnica. |
| **5. Objeción anticipada** | *"¿Por qué no SAR? ¿No es más robusto ante nubosidad?"*  Refutar: SAR-VH tiene Δ=0.188 (4º lugar), bueno pero lejos de RedEdge. |

## 2A — Versión Exploratoria (Seaborn)

In [ ]:
# ── EXPLORATORIA 2A: canales espectrales — paleta por grupo de sensor ────────
COLORES_EXP = {
    'RedEdge'    : '#DC2626',
    'Topografía' : '#F97316',
    'SAR'        : '#EAB308',
    'Óptico'     : '#3B82F6',
}

def sensor_cat(name):
    n = name.upper()
    if 'REDEDGE' in n or 'B6' in n or 'B7' in n: return 'RedEdge'
    if 'DEM' in n or 'SLOPE' in n: return 'Topografía'
    if 'VV' in n or 'VH' in n or 'SAR' in n: return 'SAR'
    return 'Óptico'

df_ch = df_channels.copy()
df_ch['Sensor'] = df_ch['Nombre'].apply(sensor_cat)
df_ch = df_ch.sort_values('|Delta|', ascending=True)

sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_facecolor('white')

bars = ax.barh(df_ch['Nombre'], df_ch['|Delta|'],
               color=[COLORES_EXP[s] for s in df_ch['Sensor']],
               alpha=0.85, edgecolor='white')
for bar, val in zip(bars, df_ch['|Delta|']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8.5)

ax.set_xlabel('Diferencia absoluta de intensidad media entre clases  |Δ|')
ax.set_ylabel('Canal espectral')
ax.set_title('[EXPLORATORIA] Poder discriminativo por canal espectral')
ax.xaxis.grid(True, color='#E5E7EB', linewidth=0.8)
ax.yaxis.grid(False)

leyenda = [mpatches.Patch(color=c, label=g) for g, c in COLORES_EXP.items()]
ax.legend(handles=leyenda, title='Grupo de sensor', fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exp_2a_canales_exploratorio.png', dpi=120, bbox_inches='tight', facecolor='white')
plt.show()

print('-> Señal clara: RedEdge3 (Delta=0.807) domina. SAR-VH (Delta=0.188) es el 4°.')
print('-> Los canales ópticos básicos (RGB, NIR) tienen Delta < 0.10: casi no discriminan.')


## 2B — Versión Aclaratoria (Plotly `go.Figure`)

**Una gráfica, un mensaje:** *"Solo 2 bandas importan — y son las RedEdge."*

In [ ]:
# ── ACLARATORIA: Plotly go.Figure — un mensaje, control total ─────────────────
ROJO   = '#D62728'
ROJO2  = '#FF6B6B'   # rojo secundario para 2° canal clave
GRIS   = '#9CA3AF'
FONDO  = '#FAFAFA'
TEXTO  = '#1F2937'

df_plot = df_channels.sort_values('|Delta|', ascending=True)

# Codificación retórica de colores:
# RedEdge3  rojo intenso (protagonista del mensaje)
# RedEdge2  rojo suave (coadyuvante)
# Resto     gris muy transparente (contexto silenciado)
TOP1 = 'S2-B7 RedEdge3'
TOP2 = 'S2-B6 RedEdge2'

colores = []
opacidades = []
for n in df_plot['Nombre']:
    if n == TOP1:   colores.append(ROJO);  opacidades.append(1.0)
    elif n == TOP2: colores.append(ROJO2); opacidades.append(0.85)
    else:           colores.append(GRIS);  opacidades.append(0.35)

# ── 1. go.Figure() ────────────────────────────────────────────────────────────
fig2 = go.Figure()

fig2.add_trace(go.Bar(
    y=df_plot['Nombre'],
    x=df_plot['|Delta|'],
    orientation='h',
    marker=dict(
        color=colores,
        opacity=opacidades,
        line=dict(width=0),
    ),
    # Etiquetas solo en los 2 canales protagonistas
    text=[f'<b>|Δ| = {v:.3f}</b>' if n in [TOP1, TOP2] else ''
          for n, v in zip(df_plot['Nombre'], df_plot['|Delta|'])],
    textposition='outside',
    textfont=dict(color=ROJO, size=13),
    hovertemplate='<b>%{y}</b><br>|Δ| = %{x:.4f}<extra></extra>',
))

# ── 2. add_annotation() — storytelling en 3 capas ────────────────────────────

# Capa 1: Mensaje principal sobre el top canal
fig2.add_annotation(
    x=0.45, y=TOP1,
    text=(
        '<b>RedEdge3 (B7): el canal que "ve" el deslizamiento</b><br>'
        'El suelo desnudo expuesto tras un deslizamiento<br>'
        'absorbe de forma distintiva en 783nm (RedEdge).<br>'
        '<b>10× más discriminativo que el canal Azul.</b>'
    ),
    showarrow=True,
    arrowhead=2,
    arrowcolor=ROJO,
    arrowwidth=2,
    ax=-10, ay=60,
    font=dict(size=11, color=ROJO),
    bgcolor='#FEF2F2',
    bordercolor=ROJO,
    borderwidth=1,
    borderpad=6,
    align='left',
    xanchor='left',
)

# Capa 2: Refutación de objeción anticipada (SAR)
fig2.add_annotation(
    x=0.19, y='S1-VH SAR',
    text=(
        '<i>SAR-VH: robusto ante nubosidad,<br>'
        'pero 4× menos discriminativo que RedEdge3</i>'
    ),
    showarrow=True,
    arrowhead=1,
    arrowcolor='#6B7280',
    ax=60, ay=-30,
    font=dict(size=10, color='#6B7280'),
    bgcolor='white',
    bordercolor='#D1D5DB',
    borderwidth=1,
    borderpad=4,
)

# Capa 3: Zona de énfasis — los 2 canales clave
fig2.add_hrect(
    y0=11.5, y1=13.5,
    fillcolor=ROJO,
    opacity=0.06,
    line_width=0,
    annotation_text='zona de alta señal',
    annotation_position='right',
    annotation_font_color=ROJO,
    annotation_font_size=10,
)

# ── 3. update_layout() — ruido cero, mensaje total ───────────────────────────
fig2.update_layout(
    title=dict(
        text=(
            '<b>Solo 2 de 14 bandas satelitales contienen la señal clave del deslizamiento</b><br>'
            '<span style="color:#6B7280;font-size:12px">'
            'Sentinel-2 RedEdge (B6, B7) · |Δ media| = diferencia Landslide − No-Landslide</span>'
        ),
        font=dict(size=15, color=TEXTO),
        x=0, xanchor='left',
    ),
    paper_bgcolor=FONDO,
    plot_bgcolor=FONDO,
    xaxis=dict(
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor='#E5E7EB',
        tickfont=dict(color='#6B7280', size=11),
        title=dict(text='|Δ media| — mayor valor = más discriminativo', font=dict(size=11, color='#6B7280')),
        range=[0, 1.05],
    ),
    yaxis=dict(
        showgrid=False,
        showline=False,
        tickfont=dict(color=TEXTO, size=11),
    ),
    showlegend=False,
    height=500,
    margin=dict(l=20, r=20, t=100, b=40),
)

fig2.show()
fig2.write_html(f'{FIG_DIR}/acl_2b_canales_aclaratorio.html')
print('Gráfica aclaratoria guardada: acl_2b_canales_aclaratorio.html')

###  Justificación de decisiones visuales — Comparativa 2

| Elemento | Exploratoria | Aclaratoria | Principio |
|---|---|---|---|
| **Paleta** | 4 colores por tipo de sensor | 1 rojo + gris | Pre-atención: el color no clasifica, **señala** |
| **Opacidad** | Uniforme (0.8) | Variable: 1.0 / 0.85 / 0.35 | Jerarquía sin eliminar contexto |
| **`add_hrect()`** | — | Zona sombreada sobre top 2 | Agrupa visualmente el hallazgo antes de leer texto |
| **Anotaciones** | — | 3 capas: mensaje + causa física + objeción | Cada `add_annotation()` tiene función retórica distinta |
| **Etiquetas** | Todos los valores | Solo top 2 en negrita | Reducción de carga cognitiva |
| **`showgrid=False`** | Grid oscuro visible | Sin grid | Tufte: cada pixel de tinta debe justificarse |
| **Título** | `"Poder discriminativo por canal"` | `"Solo 2 de 14 bandas..."` | Cuantifica el hallazgo — urgencia inmediata |

---
#  Comparativa 3: Trade-off Precisión vs. Recall

## Paso 0: Las 5 preguntas

| Pregunta | Respuesta |
|---|---|
| **1. Audiencia** | Directivos de emergencias / autoridades de gestión de riesgo. Sin experiencia técnica en ML. |
| **2. Mensaje único** | *"Para salvar vidas, necesitamos un modelo que no falle en detectar deslizamientos reales, aunque genere falsas alarmas. Random Forest logra Recall=95.7%."* |
| **3. Acción deseada** | Aprobar el despliegue de Random Forest como sistema de alerta temprana. |
| **4. Tiempo disponible** | < 20 segundos. El título debe ser suficiente para tomar la decisión. |
| **5. Objeción anticipada** | *"Si genera muchas falsas alarmas (baja precisión 74%), la gente deja de hacer caso."*  Mostrar que el trade-off es razonable y que 95.7% de Recall significa casi ningún deslizamiento sin detectar. |

## 3A — Versión Exploratoria (Matplotlib)

In [ ]:
# ── EXPLORATORIA 3A: Precisión vs. Recall — scatter exploratorio ─────────────
# Datos con conversión numérica correcta (columnas con '—' -> NaN)
datos_pr = [
    {'Modelo':'Logistic Reg.', 'Precisión':0.7971, 'Recall':0.7806, 'F1':0.7886, 'Tipo':'Clásico'},
    {'Modelo':'SVM (RBF)',     'Precisión':0.8193, 'Recall':0.7777, 'F1':0.7974, 'Tipo':'Clásico'},
    {'Modelo':'Random Forest', 'Precisión':0.7439, 'Recall':0.9569, 'F1':0.8368, 'Tipo':'Clásico'},
    {'Modelo':'ResNet-50',     'Precisión':0.7219, 'Recall':0.8771, 'F1':0.7865, 'Tipo':'Deep Learning'},
]
df_c = pd.DataFrame(datos_pr)

sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(8, 6))
ax.set_facecolor('white')

colores_exp = {'Clásico':'#2CA02C', 'Deep Learning':'#7C3AED'}
for tipo, g in df_c.groupby('Tipo'):
    ax.scatter(g['Precisión'], g['Recall'],
               c=colores_exp[tipo], s=g['F1']*350,
               label=tipo, alpha=0.85, edgecolors='white', linewidth=1.5, zorder=5)
    for _, row in g.iterrows():
        ax.annotate(row['Modelo'], (row['Precisión'], row['Recall']),
                    textcoords='offset points', xytext=(7, 4), fontsize=9)

ax.set_xlabel('Precisión  (fracción de alertas correctas)')
ax.set_ylabel('Recall  (fracción de deslizamientos detectados)')
ax.set_title('[EXPLORATORIA] Precisión vs. Recall — ¿dónde están los modelos?')
ax.legend()
ax.set_xlim(0.65, 0.90)
ax.set_ylim(0.70, 1.02)
ax.yaxis.grid(True, color='#E5E7EB', linewidth=0.8)
ax.xaxis.grid(True, color='#E5E7EB', linewidth=0.8)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exp_3a_prec_recall_exploratorio.png', dpi=120, bbox_inches='tight', facecolor='white')
plt.show()

print('-> Descubrimiento: RF tiene el recall más alto (0.957) pero la menor precisión (0.744).')
print('-> Para alertas tempranas, el recall manda sobre la precisión.')


## 3B — Versión Aclaratoria (Plotly `go.Figure`)

**Una gráfica, un mensaje:** *"Para sistemas de alerta, el Recall = 95.7% es la métrica que importa."*

In [ ]:
# ── ACLARATORIA: Scatter Plotly go.Figure — decisión de negocio ───────────────
ROJO   = '#D62728'
GRIS   = '#9CA3AF'
FONDO  = '#FAFAFA'
TEXTO  = '#1F2937'

datos_pr3 = [
    {'Modelo':'Logistic Reg.', 'Precisión':0.7971, 'Recall':0.7806, 'F1':0.7886, 'Tipo':'Clásico'},
    {'Modelo':'SVM (RBF)',     'Precisión':0.8193, 'Recall':0.7777, 'F1':0.7974, 'Tipo':'Clásico'},
    {'Modelo':'Random Forest', 'Precisión':0.7439, 'Recall':0.9569, 'F1':0.8368, 'Tipo':'Clásico'},
    {'Modelo':'ResNet-50',     'Precisión':0.7219, 'Recall':0.8771, 'F1':0.7865, 'Tipo':'Deep Learning'},
]
df_c = pd.DataFrame(datos_pr3)

# ── 1. go.Figure() — figura base con curvas iso-F1 como contexto ──────────────
fig3 = go.Figure()

# Curvas iso-F1: contexto técnico en gris muy suave
for f1_val in [0.75, 0.80, 0.85]:
    p = np.linspace(0.65, 0.92, 300)
    r = f1_val * p / (2*p - f1_val + 1e-9)
    mask = (r >= 0.70) & (r <= 1.0)
    fig3.add_trace(go.Scatter(
        x=p[mask], y=r[mask],
        mode='lines',
        line=dict(color='#E5E7EB', width=1.2, dash='dot'),
        showlegend=False,
        hoverinfo='skip',
        name=f'iso-F1={f1_val}',
    ))
    # Etiqueta iso-F1: discreta, no compite
    mid = mask.sum() // 2
    fig3.add_annotation(
        x=p[mask][mid]+0.004, y=r[mask][mid],
        text=f'F1={f1_val}', showarrow=False,
        font=dict(size=9, color='#D1D5DB'),
    )

# Zona de alta alerta: Recall > 0.90 — zona donde queremos estar
fig3.add_hrect(
    y0=0.90, y1=1.02,
    fillcolor=ROJO, opacity=0.05, line_width=0,
)

# Modelos no protagonistas — puntos pequeños y grises
df_otros = df_c[df_c['Modelo'] != 'Random Forest']
fig3.add_trace(go.Scatter(
    x=df_otros['Precisión'], y=df_otros['Recall'],
    mode='markers+text',
    marker=dict(color=GRIS, size=10, opacity=0.5,
                line=dict(color='white', width=1)),
    text=df_otros['Modelo'],
    textposition='top right',
    textfont=dict(size=10, color='#9CA3AF'),
    showlegend=False,
    hovertemplate='<b>%{text}</b><br>Precisión: %{x:.3f}<br>Recall: %{y:.3f}<extra></extra>',
))

# Random Forest — protagonista rojo y grande
rf = df_c[df_c['Modelo'] == 'Random Forest'].iloc[0]
fig3.add_trace(go.Scatter(
    x=[rf['Precisión']], y=[rf['Recall']],
    mode='markers',
    marker=dict(color=ROJO, size=22, symbol='circle',
                line=dict(color='white', width=2)),
    showlegend=False,
    hovertemplate=(
        '<b>Random Forest</b><br>'
        'Precisión: 0.744 (74.4% de alertas correctas)<br>'
        'Recall: 0.957 (95.7% de deslizamientos detectados)<br>'
        'F1: 0.837<extra></extra>'
    ),
))

# ── 2. add_annotation() — 3 capas retóricas ──────────────────────────────────

# Capa 1: Mensaje principal — por qué Recall manda
fig3.add_annotation(
    x=rf['Precisión'], y=rf['Recall'],
    text=(
        '<b>Recall = 95.7%</b><br>'
        'De cada 100 deslizamientos reales,<br>'
        'el modelo detecta 96.<br>'
        '<i>Solo 4 pasan desapercibidos.</i>'
    ),
    showarrow=True,
    arrowhead=2, arrowcolor=ROJO, arrowwidth=2,
    ax=-120, ay=60,
    font=dict(size=11, color=ROJO),
    bgcolor='#FEF2F2',
    bordercolor=ROJO,
    borderwidth=1, borderpad=6,
    align='left',
)

# Capa 2: Refutación de objeción sobre baja precisión
fig3.add_annotation(
    x=rf['Precisión'], y=0.76,
    text=(
        '<i>Precisión = 74.4%: por cada 4 alertas,<br>'
        '3 son deslizamientos reales. Aceptable<br>'
        'para sistemas preventivos.</i>'
    ),
    showarrow=False,
    font=dict(size=10, color='#6B7280'),
    bgcolor='white',
    bordercolor='#D1D5DB',
    borderwidth=1, borderpad=4,
    align='left', xanchor='center',
)

# Capa 3: Etiqueta zona alta alerta
fig3.add_annotation(
    x=0.655, y=0.91,
    text='<b>ZONA DE ALTA ALERTA</b><br><i>Recall > 90%: mínimo requerido para alertas tempranas</i>',
    showarrow=False,
    font=dict(size=10, color=ROJO),
    xanchor='left', yanchor='bottom',
)

# ── 3. update_layout() — ejes en lenguaje de negocio, no estadístico ─────────
fig3.update_layout(
    title=dict(
        text=(
            '<b>Para alertas tempranas, el Recall es la métrica crítica — y Random Forest lidera</b><br>'
            '<span style="color:#6B7280;font-size:12px">'
            'Un falso negativo = deslizamiento no detectado = vidas en riesgo</span>'
        ),
        font=dict(size=14, color=TEXTO),
        x=0, xanchor='left',
    ),
    paper_bgcolor=FONDO,
    plot_bgcolor=FONDO,
    xaxis=dict(
        range=[0.64, 0.91],
        showgrid=False,
        showline=True, linecolor='#E5E7EB',
        tickformat='.0%',
        tickfont=dict(color='#6B7280', size=11),
        # Eje en lenguaje de negocio
        title=dict(
            text='Precisión — ¿qué fracción de las alertas son deslizamientos reales?',
            font=dict(size=11, color='#6B7280')
        ),
    ),
    yaxis=dict(
        range=[0.70, 1.03],
        showgrid=False,
        showline=True, linecolor='#E5E7EB',
        tickformat='.0%',
        tickfont=dict(color='#6B7280', size=11),
        title=dict(
            text='Recall — ¿qué fracción de los deslizamientos reales detectamos?',
            font=dict(size=11, color='#6B7280')
        ),
    ),
    showlegend=False,
    height=480,
    margin=dict(l=20, r=20, t=100, b=60),
)

fig3.show()
fig3.write_html(f'{FIG_DIR}/acl_3b_prec_recall_aclaratorio.html')
print('Gráfica aclaratoria guardada: acl_3b_prec_recall_aclaratorio.html')

###  Justificación de decisiones visuales — Comparativa 3

| Elemento | Exploratoria | Aclaratoria | Principio |
|---|---|---|---|
| **Ejes** | `"Precisión"` / `"Recall"` | `"¿qué fracción de alertas son correctas?"` | Lenguaje de negocio: el gerente no sabe qué es Recall |
| **Escala** | Decimal (0.744) | Porcentaje (74.4%) | Conversión al lenguaje natural del decisor |
| **Zona sombreada** | — | `add_hrect()` Recall>90% | Ubica visualmente el umbral mínimo aceptable |
| **Anotación 1** | — | Traduce `Recall=0.957`  `"Solo 4 de cada 100 pasan desapercibidos"` | Consecuencia humana, no estadística |
| **Anotación 2** | — | Refuta objeción de baja precisión | Anticipa la pregunta del decisor |
| **Iso-F1 curves** | — | Presentes pero en `#E5E7EB` (casi invisibles) | Contexto técnico sin competir con el mensaje |
| **Tamaño RF** | `s=F1*300` (uniforme entre modelos) | `size=22` (3× mayor que el resto) | RF es el protagonista —tamaño = importancia |

---
##  Resumen de Patrones Plotly Aclaratorio

```python
# ═══════════════════════════════════════════════════════════════
# PATRÓN ESTÁNDAR: Gráfica Aclaratoria con Plotly
# ═══════════════════════════════════════════════════════════════

import plotly.graph_objects as go

fig = go.Figure()                           # 1. Canvas en blanco

fig.add_trace(go.Bar(...))                  # 2. Datos con codificación retórica
#   marker=dict(color=[ROJO, GRIS, GRIS])   #    UN color de énfasis, resto neutro
#   opacity=[1.0, 0.4, 0.4]                 #    Opacidad como jerarquía

fig.add_annotation(                         # 3. Storytelling integrado
    text='El mensaje que el directivo debe recordar',
    showarrow=True,                         #    Flecha apunta al dato exacto
    bgcolor='#FEF2F2',                      #    Fondo suave del color de énfasis
    bordercolor=ROJO,                       #    Borde del mismo color
)                                           #     Cada anotación tiene función retórica

fig.update_layout(                          # 4. Eliminar todo el ruido visual
    title='<b>Título como argumento</b>',   #    Título = conclusión, no descripción
    paper_bgcolor='#FAFAFA',               #    Fondo neutro
    plot_bgcolor='#FAFAFA',
    xaxis=dict(showgrid=False),            #    Sin grid
    yaxis=dict(showgrid=False),
    showlegend=False,                      #    Sin leyenda cuando el color ya es el mensaje
)
```

> **Regla de oro (Tufte, 1983):**  
> *Data-to-ink ratio = (tinta que muestra datos) / (tinta total)*  
> El análisis aclaratorio maximiza este ratio eliminando grids, leyendas redundantes, bordes decorativos y cualquier elemento que no transmita el mensaje central.